# 03 - MLP với PyTorch

Các điểm đã cải tiến so với bản cũ:
- **Chuẩn hóa nhãn** (trừ mean, chia std) -> mạng không còn phải "học" từ giá trị ~12 (RMSE epoch 1 trước đây là 8.09).
- **StandardScaler fit trong từng fold** (không rò rỉ), chỉ scale cột liên tục, giữ nguyên cột One-Hot 0/1.
- **Early stopping + lưu best model** (`best_val_loss` trước đây khai báo nhưng không dùng) + `ReduceLROnPlateau`.
- **Cùng 5-fold (seed 42) với sklearn** -> so sánh công bằng; dự đoán test = trung bình 5 mô hình theo fold.
- Thử vài cấu hình siêu tham số và chọn theo CV.

In [1]:
import os, copy, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler
warnings.filterwarnings("ignore")

FEATURE_SET = "clean"      # 'clean' (notebook 01) hoặc 'best' (notebook 04) - giống notebook 02
SEED = 42
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

X_df = pd.read_csv(f"../data/X_train_{FEATURE_SET}.csv")
X_test_df = pd.read_csv(f"../data/X_test_{FEATURE_SET}.csv")
X = X_df.values.astype(np.float32)
X_test = X_test_df.values.astype(np.float32)
y = pd.read_csv("../data/y_train_clean.csv")["SalePrice"].values.astype(np.float32)
sample_submission = pd.read_csv("../data/sample_submission.csv")

# Cột liên tục = cột có > 2 giá trị khác nhau (cột One-Hot/flag 0/1 giữ nguyên)
cont_idx = np.where(X_df.nunique().values > 2)[0]

os.makedirs("../results", exist_ok=True); os.makedirs("../submissions", exist_ok=True)
print("Thiết bị:", device, "| Bộ đặc trưng:", FEATURE_SET)
print("X:", X.shape, "| X_test:", X_test.shape, "| số cột liên tục:", len(cont_idx))

Thiết bị: cpu | Bộ đặc trưng: clean
X: (1458, 306) | X_test: (1459, 306) | số cột liên tục: 39


## 1. Kiến trúc mô hình

In [2]:
class HousePriceMLP(nn.Module):
    def __init__(self, input_dim, hidden=(128, 64, 32), dropout=0.3):
        super().__init__()
        layers, prev = [], input_dim
        for i, h in enumerate(hidden):
            layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(),
                       nn.Dropout(dropout if i < len(hidden) - 1 else 0.0)]
            prev = h
        layers.append(nn.Linear(prev, 1))        # đầu ra: log(SalePrice) đã chuẩn hóa
        self.network = nn.Sequential(*layers)

    def forward(self, x):
        return self.network(x).squeeze(1)

print(HousePriceMLP(X.shape[1]))

HousePriceMLP(
  (network): Sequential(
    (0): Linear(in_features=306, out_features=128, bias=True)
    (1): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3, inplace=False)
    (8): Linear(in_features=64, out_features=32, bias=True)
    (9): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.0, inplace=False)
    (12): Linear(in_features=32, out_features=1, bias=True)
  )
)


## 2. Huấn luyện một fold
Early stopping theo RMSE trên fold validation. *Lưu ý:* vì epoch tốt nhất được chọn trên chính fold validation nên RMSE báo cáo hơi lạc quan - tương tự việc chọn siêu tham số ở notebook 02.

In [3]:
@torch.no_grad()
def predict(model, A):
    model.eval()
    return model(torch.tensor(A, dtype=torch.float32, device=device)).cpu().numpy()

def train_one_fold(X_tr, y_tr, X_val, y_val, X_te, cfg):
    set_seed(cfg["seed"])

    # Chuẩn hóa feature liên tục (fit trên train của fold) & chuẩn hóa nhãn
    scaler = StandardScaler().fit(X_tr[:, cont_idx])
    def prep(A):
        A = A.copy(); A[:, cont_idx] = scaler.transform(A[:, cont_idx]); return A
    X_tr_s, X_val_s, X_te_s = prep(X_tr), prep(X_val), prep(X_te)
    y_mean, y_std = float(y_tr.mean()), float(y_tr.std())
    y_tr_s = (y_tr - y_mean) / y_std
    y_val_s = (y_val - y_mean) / y_std

    loader = DataLoader(TensorDataset(torch.tensor(X_tr_s), torch.tensor(y_tr_s, dtype=torch.float32)),
                        batch_size=cfg["batch_size"], shuffle=True, drop_last=True)
    Xv = torch.tensor(X_val_s, device=device); yv = torch.tensor(y_val_s, dtype=torch.float32, device=device)

    model = HousePriceMLP(X_tr.shape[1], cfg["hidden"], cfg["dropout"]).to(device)
    criterion = nn.MSELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=cfg["weight_decay"])
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=10)

    best_val, best_state, wait, history = float("inf"), None, 0, []
    for epoch in range(1, cfg["epochs"] + 1):
        model.train(); total, seen = 0.0, 0
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()
            total += loss.item() * xb.size(0); seen += xb.size(0)
        train_rmse = np.sqrt(total / seen) * y_std              # quy về thang log giá

        model.eval()
        with torch.no_grad():
            val_rmse = float(torch.sqrt(criterion(model(Xv), yv))) * y_std
        scheduler.step(val_rmse)
        history.append((train_rmse, val_rmse))

        if val_rmse < best_val - 1e-5:
            best_val, best_state, wait = val_rmse, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= cfg["patience"]:
                break

    model.load_state_dict(best_state)                            # quay lại epoch tốt nhất
    val_pred = predict(model, X_val_s) * y_std + y_mean          # thang log giá
    test_pred = predict(model, X_te_s) * y_std + y_mean
    return best_val, val_pred, test_pred, history

def run_cv(cfg, verbose=True):
    kf = KFold(n_splits=5, shuffle=True, random_state=42)       # GIỐNG notebook 02
    oof = np.zeros(len(y)); fold_rmse, test_preds, histories = [], [], []
    for k, (tr, va) in enumerate(kf.split(X), 1):
        r, vp, tp, h = train_one_fold(X[tr], y[tr], X[va], y[va], X_test, cfg)
        oof[va] = vp; fold_rmse.append(r); test_preds.append(tp); histories.append(h)
        if verbose: print(f"  fold {k}: RMSE = {r:.4f} ({len(h)} epochs)")
    return {"oof": oof, "fold_rmse": np.array(fold_rmse), "test_log": np.mean(test_preds, axis=0), "histories": histories}

## 3. Thử các cấu hình & chọn theo cross-validation

In [ ]:
base_cfg = dict(batch_size=32, epochs=300, patience=40, seed=SEED)
configs = {
    "A: 128-64-32, drop 0.2, wd 1e-2": dict(hidden=(128, 64, 32), dropout=0.2, lr=1e-3, weight_decay=1e-2),
    "B: 128-64-32, drop 0.3, wd 1e-2": dict(hidden=(128, 64, 32), dropout=0.3, lr=1e-3, weight_decay=1e-2),
    "C: 256-128-64, drop 0.3, wd 5e-2": dict(hidden=(256, 128, 64), dropout=0.3, lr=1e-3, weight_decay=5e-2),
    "D: 64-32, drop 0.3, wd 1e-2":     dict(hidden=(64, 32), dropout=0.3, lr=1e-3, weight_decay=1e-2),
}

runs, rows = {}, []
for name, c in configs.items():
    print(name)
    runs[name] = run_cv({**base_cfg, **c})
    r = runs[name]
    rows.append({"Cấu hình": name, "CV RMSE (mean)": r["fold_rmse"].mean(), "CV RMSE (std)": r["fold_rmse"].std(),
                 "OOF RMSE": float(np.sqrt(np.mean((y - r["oof"]) ** 2)))})

mlp_df = pd.DataFrame(rows).sort_values("OOF RMSE").reset_index(drop=True)
display(mlp_df.round(4))
best_name = mlp_df.loc[0, "Cấu hình"]
best_run = runs[best_name]
print("=> Cấu hình MLP tốt nhất:", best_name)

A: 128-64-32, drop 0.2, wd 1e-2
  fold 1: RMSE = 0.1181 (63 epochs)
  fold 2: RMSE = 0.1137 (47 epochs)
  fold 3: RMSE = 0.1212 (52 epochs)
  fold 4: RMSE = 0.1221 (45 epochs)
  fold 5: RMSE = 0.1017 (47 epochs)
B: 128-64-32, drop 0.3, wd 1e-2
  fold 1: RMSE = 0.1175 (74 epochs)
  fold 2: RMSE = 0.1143 (58 epochs)
  fold 3: RMSE = 0.1214 (53 epochs)
  fold 4: RMSE = 0.1236 (56 epochs)
  fold 5: RMSE = 0.1025 (64 epochs)
C: 256-128-64, drop 0.3, wd 5e-2
  fold 1: RMSE = 0.1181 (83 epochs)
  fold 2: RMSE = 0.1134 (50 epochs)
  fold 3: RMSE = 0.1205 (48 epochs)
  fold 4: RMSE = 0.1232 (46 epochs)


In [ ]:
# Đường học (fold 1) của cấu hình tốt nhất
h = np.array(best_run["histories"][0])
plt.figure(figsize=(7, 4))
plt.plot(h[:, 0], label="Train RMSE"); plt.plot(h[:, 1], label="Val RMSE")
plt.xlabel("Epoch"); plt.ylabel("RMSE (log giá)"); plt.title(f"Learning curve - {best_name} (fold 1)")
plt.ylim(0, min(1.0, h.max())); plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

## 4. Xuất file nộp & so sánh với sklearn

In [ ]:
final_test_log = best_run["test_log"]          # trung bình 5 mô hình theo fold
submission_mlp = pd.DataFrame({"Id": sample_submission["Id"], "SalePrice": np.expm1(final_test_log)})
assert len(submission_mlp) == len(sample_submission) and submission_mlp["SalePrice"].notna().all() and (submission_mlp["SalePrice"] > 0).all()
submission_mlp.to_csv("../submissions/submission_mlp.csv", index=False)

np.save("../results/oof_mlp.npy", best_run["oof"])
np.save("../results/test_pred_mlp_log.npy", final_test_log)
mlp_df.to_csv("../results/mlp_cv.csv", index=False)
print("Đã xuất '../submissions/submission_mlp.csv'")

# So sánh với kết quả sklearn (notebook 02) - cùng bộ fold
oof_mlp_rmse = float(np.sqrt(np.mean((y - best_run["oof"]) ** 2)))
compare = [{"Mô hình": f"MLP PyTorch ({best_name[0]})", "OOF RMSE": oof_mlp_rmse}]
if os.path.exists("../results/sklearn_cv.csv"):
    sk = pd.read_csv("../results/sklearn_cv.csv")[["Mô hình", "OOF RMSE"]]
    compare = sk.to_dict("records") + compare
display(pd.DataFrame(compare).sort_values("OOF RMSE").reset_index(drop=True).round(4))

## 5. (Tùy chọn) Blend sklearn + MLP

In [ ]:
if os.path.exists("../results/oof_sklearn.npy"):
    oof_sk = np.load("../results/oof_sklearn.npy"); test_sk = np.load("../results/test_pred_sklearn_log.npy")
    ws = [round(float(w), 1) for w in np.linspace(0, 1, 11)]
    sc = {w: float(np.sqrt(np.mean((y - ((1 - w) * oof_sk + w * best_run["oof"])) ** 2))) for w in ws}
    w_best = min(sc, key=sc.get)
    print("OOF RMSE theo trọng số MLP:", {k: round(v, 4) for k, v in sc.items()})
    print(f"=> w_MLP tốt nhất = {w_best} | OOF RMSE = {sc[w_best]:.4f}")

    blend_log = (1 - w_best) * test_sk + w_best * final_test_log
    sub_blend = pd.DataFrame({"Id": sample_submission["Id"], "SalePrice": np.expm1(blend_log)})
    sub_blend.to_csv("../submissions/submission_blend.csv", index=False)
    print("Đã xuất '../submissions/submission_blend.csv'")
else:
    print("Chưa có ../results/oof_sklearn.npy - hãy chạy notebook 02 trước.")